# SequenceVis

Converts a CSV matrix of position-specific amino acid frequencies into foreground/background sequence data for visualization.

## Overview

This notebook takes a CSV table of amino acid probabilities at positions -5 to +4 (relative to a motif center) and:
1. **Foreground (fg)**: Converts probabilities into a discrete pool of sequences enriched for the motif
2. **Background (bg)**: Generates random sequences with a constraint at position 0 (only S or T)

## Workflow
1. Fetch source modules directly from GitHub
2. Generate a random input CSV (or upload your own)
3. Convert the CSV into foreground/background datasets
4. Export results as `fg.csv` and `bg.csv`

## Setup

Install required dependencies and fetch source modules from GitHub.

In [ ]:
# Install required packages
!pip install pandas numpy -q

In [ ]:
import pandas as pd
import numpy as np
import sys, os, subprocess

## Fetch Source Modules from GitHub

The cells below download the source `.py` files directly from the [sequenceVis GitHub repository](https://github.com/CrankyTitanO7/sequenceVis).

This means the notebook **always stays in sync** with the source code — when the repo is updated, just re-run these cells to get the latest version.

In [ ]:
# Fetch biology.py (amino acid alphabet)
!wget -q -O biology.py https://raw.githubusercontent.com/CrankyTitanO7/sequenceVis/main/biology.py
!echo 'biology.py:' && head -3 biology.py

In [ ]:
# Fetch csv-to-fgbg.py (core conversion logic)
!wget -q -O csv_to_fgbg.py https://raw.githubusercontent.com/CrankyTitanO7/sequenceVis/main/csv-to-fgbg.py
!echo 'csv_to_fgbg.py:' && wc -l csv_to_fgbg.py

In [ ]:
# Fetch random_input.py (synthetic data generator)
!wget -q -O random_input.py https://raw.githubusercontent.com/CrankyTitanO7/sequenceVis/main/random_input.py
!echo 'random_input.py:' && wc -l random_input.py

In [ ]:
# Import the fetched modules
import biology
print('biology loaded:', biology.AMINO_ALPH[:3], '...')

import csv_to_fgbg
print('csv_to_fgbg loaded')
print('  Functions:', [x for x in dir(csv_to_fgbg) if not x.startswith('_')])

import random_input
print('random_input loaded')

## 1. Generate Random Input CSV

Creates a synthetic 20×10 matrix (20 amino acids × 10 positions: -5 through +4) with random values normalized so each column sums to 20 (i.e., probabilities out of 20).

In [ ]:
# Re-run the random_input script to generate a fresh input.csv
import importlib
importlib.reload(random_input)

df = pd.read_csv('input.csv')
print("input.csv generated successfully")
df.head()

### (Optional) Upload Your Own CSV

If you have a custom CSV file, run the cell below to upload it. It will overwrite the generated `input.csv`.

The expected format is a CSV with a `titles` column (amino acid one-letter codes) and numeric columns for each position.

In [ ]:
from google.colab import files

uploaded = files.upload()
for fn in uploaded.keys():
  print(f'Uploaded file: {fn}')
  # Copy the uploaded file to input.csv so the pipeline picks it up
  import shutil
  shutil.copy(fn, 'input.csv')

## 2. Run the Conversion

Execute the foreground and background generators from the fetched source modules.

- **Foreground (fg)**: Each position's amino acid probabilities are converted into a discrete pool of 1000 amino acids. Then all 10 positions are compiled into a single sequence string.
- **Background (bg)**: 5000 random 10-mer sequences are generated, with position 0 constrained to only S or T.

Output:
- `fg_full.csv` — full foreground with all columns (sequence + each position)
- `fg.csv` — just the compiled sequence column
- `bg.csv` — background sequences

In [ ]:
# Override the hardcoded PATH to use the notebook's input.csv
csv_to_fgbg.PATH = "input.csv"

fg_dat = csv_to_fgbg.fg_generate_compiled_sequence()
print("////////////// foreground generator tests //////////////")
fg_dat.head()

In [ ]:
bg_dat = csv_to_fgbg.bg()
print("\n////////////// background generator tests //////////////")
print(bg_dat["bg"].head(10).tolist(), end="... and etc\n")
print("//////////////       tests  concluded     //////////////\n")

## 3. Export Results

Save the foreground and background data to CSV files and download them.

In [ ]:
bg_dat.to_csv("bg.csv", index=False)
fg_dat.to_csv("fg_full.csv", index=False)
fg_dat.iloc[:, [0]].to_csv("fg.csv", index=False)
print("Exported fg.csv, fg_full.csv, and bg.csv")

In [ ]:
from google.colab import files

files.download("fg.csv")
files.download("bg.csv")

## Summary

- **`fg.csv`**: Foreground dataset — one column with compiled 10-mer sequences. Each sequence's amino acids at each position match the input probabilities scaled to 1000 samples.
- **`fg_full.csv`**: Same foreground data but with all position columns (-5 to +4) included for inspection.
- **`bg.csv`**: Background dataset — 5000 random 10-mer sequences with position 0 constrained to S/T.

These files can be used for downstream motif visualization (e.g., sequence logos, position weight matrices, etc.).

---

> **Note**: The source modules (`biology.py`, `csv_to_fgbg.py`, `random_input.py`) are fetched fresh from GitHub each time the setup cells are re-run. This keeps the notebook automatically in sync with the repository.